# Hospital Data Cleaning & Transformation
**MedTrack_DV — Module 2**

This notebook cleans and standardizes the 4 raw hospital datasets:
- `patients.csv`
- `staff.csv`
- `staff_schedule.csv`
- `services_weekly.csv`

**Tasks covered:** remove duplicates, handle missing data, standardize `service` (department) names, normalize date/indicator formats, and export Tableau-ready CSVs.

**Common linking key across all files:** `service` (department). `staff_id` links `staff` ↔ `staff_schedule`. `week` links `staff_schedule` ↔ `services_weekly`.

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

pd.set_option('display.max_columns', None)

## 1. Load raw datasets

In [2]:
patients = pd.read_csv(f"{RAW_DIR}/patients.csv")
staff = pd.read_csv(f"{RAW_DIR}/staff.csv")
staff_schedule = pd.read_csv(f"{RAW_DIR}/staff_schedule.csv")
services_weekly = pd.read_csv(f"{RAW_DIR}/services_weekly.csv")

print("patients:", patients.shape)
print("staff:", staff.shape)
print("staff_schedule:", staff_schedule.shape)
print("services_weekly:", services_weekly.shape)

patients: (1000, 7)
staff: (110, 4)
staff_schedule: (6552, 6)
services_weekly: (208, 10)


## 2. Initial data quality check (before cleaning)

In [3]:
def quality_report(df, name):
    print(f"--- {name} ---")
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values per column:")
    print(df.isnull().sum())
    print("Missing %% overall: %.2f%%" % (df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100))
    print()

for df, name in [(patients, "patients"), (staff, "staff"),
                  (staff_schedule, "staff_schedule"), (services_weekly, "services_weekly")]:
    quality_report(df, name)

--- patients ---
Duplicate rows: 0
Missing values per column:
patient_id        0
name              0
age               0
arrival_date      0
departure_date    0
service           0
satisfaction      0
dtype: int64
Missing % overall: 0.00%

--- staff ---
Duplicate rows: 0
Missing values per column:
staff_id      0
staff_name    0
role          0
service       0
dtype: int64
Missing % overall: 0.00%

--- staff_schedule ---
Duplicate rows: 0
Missing values per column:
week          0
staff_id      0
staff_name    0
role          0
service       0
present       0
dtype: int64
Missing % overall: 0.00%

--- services_weekly ---
Duplicate rows: 0
Missing values per column:
week                    0
month                   0
service                 0
available_beds          0
patients_request        0
patients_admitted       0
patients_refused        0
patient_satisfaction    0
staff_morale            0
event                   0
dtype: int64
Missing % overall: 0.00%



## 3. Standardize `service` (department) names
Same standardization function is applied across all 4 tables so the join key matches exactly (no case/whitespace mismatches).

In [4]:
def standardize_service(series):
    return (series.astype(str)
                  .str.strip()
                  .str.lower()
                  .str.replace(r'\s+', '_', regex=True)
                  .str.replace('generalmedicine', 'general_medicine'))

patients['service'] = standardize_service(patients['service'])
staff['service'] = standardize_service(staff['service'])
staff_schedule['service'] = standardize_service(staff_schedule['service'])
services_weekly['service'] = standardize_service(services_weekly['service'])

# Sanity check: same set of department names across all 4 tables
print("patients services:", sorted(patients['service'].unique()))
print("staff services:", sorted(staff['service'].unique()))
print("staff_schedule services:", sorted(staff_schedule['service'].unique()))
print("services_weekly services:", sorted(services_weekly['service'].unique()))

patients services: ['emergency', 'general_medicine', 'icu', 'surgery']
staff services: ['emergency', 'general_medicine', 'icu', 'surgery']
staff_schedule services: ['emergency', 'general_medicine', 'icu', 'surgery']
services_weekly services: ['emergency', 'general_medicine', 'icu', 'surgery']


## 4. Clean `patients.csv`

In [5]:
# Remove exact duplicate rows
patients = patients.drop_duplicates()
patients = patients.drop_duplicates(subset="patient_id", keep="first")

print(patients["arrival_date"].head())
print(patients["arrival_date"].dtype)

patients["arrival_date"] = pd.to_datetime(
    patients["arrival_date"], dayfirst=True, errors="coerce"
)
patients["departure_date"] = pd.to_datetime(
    patients["departure_date"], dayfirst=True, errors="coerce"
)

print("arrival_date parse failures:", patients["arrival_date"].isna().sum())
print("departure_date parse failures:", patients["departure_date"].isna().sum())

bad_dates = patients["departure_date"] < patients["arrival_date"]
print("Rows with departure before arrival:", bad_dates.sum())
patients = patients[~bad_dates]

patients["length_of_stay"] = (
    patients["departure_date"] - patients["arrival_date"]
).dt.days

patients["age"] = patients["age"].fillna(patients["age"].median())
patients["satisfaction"] = patients["satisfaction"].fillna(
    patients["satisfaction"].median()
)
patients = patients.dropna(subset=["patient_id", "arrival_date", "service"])

print("Cleaned patients shape:", patients.shape)
patients.head()

0    2025-03-16
1    2025-12-13
2    2025-06-29
3    2025-10-12
4    2025-02-18
Name: arrival_date, dtype: object
object
arrival_date parse failures: 0
departure_date parse failures: 0
Rows with departure before arrival: 0
Cleaned patients shape: (1000, 8)


C:\Users\memyo\AppData\Local\Temp\ipykernel_27268\385898210.py:8: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  patients["arrival_date"] = pd.to_datetime(
C:\Users\memyo\AppData\Local\Temp\ipykernel_27268\385898210.py:11: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  patients["departure_date"] = pd.to_datetime(


,patient_id,name,age,arrival_date,departure_date,service,satisfaction,length_of_stay
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,6
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,1
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83,6
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81,11
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,icu,76,7


## 5. Clean `staff.csv`

In [6]:
staff = staff.drop_duplicates()
staff = staff.drop_duplicates(subset='staff_id', keep='first')

# Standardize role text
staff['role'] = staff['role'].astype(str).str.strip().str.lower()

staff = staff.dropna(subset=['staff_id', 'service', 'role'])

print("Cleaned staff shape:", staff.shape)
staff.head()

Cleaned staff shape: (110, 4)


,staff_id,staff_name,role,service
0,STF-5ca26577,Allison Hill,doctor,emergency
1,STF-02ae59ca,Noah Rhodes,doctor,emergency
2,STF-d8006e7c,Angie Henderson,doctor,emergency
3,STF-212d8b31,Daniel Wagner,doctor,emergency
4,STF-107a58e4,Cristian Santos,doctor,emergency


## 6. Clean `staff_schedule.csv`

In [7]:
staff_schedule = staff_schedule.drop_duplicates()

# A staff member should appear once per week -> dedupe on (week, staff_id)
staff_schedule = staff_schedule.drop_duplicates(subset=['week', 'staff_id'], keep='first')

# Normalize 'present' to 0/1 integer
staff_schedule['present'] = pd.to_numeric(staff_schedule['present'], errors='coerce').fillna(0).astype(int)
staff_schedule['week'] = pd.to_numeric(staff_schedule['week'], errors='coerce').astype('Int64')

staff_schedule = staff_schedule.dropna(subset=['staff_id', 'week', 'service'])

print("Cleaned staff_schedule shape:", staff_schedule.shape)
staff_schedule.head()

Cleaned staff_schedule shape: (6552, 6)


,week,staff_id,staff_name,role,service,present
0,1,STF-b77cdc60,Allison Hill,doctor,emergency,1
1,2,STF-b77cdc60,Allison Hill,doctor,emergency,1
2,3,STF-b77cdc60,Allison Hill,doctor,emergency,0
3,4,STF-b77cdc60,Allison Hill,doctor,emergency,1
4,5,STF-b77cdc60,Allison Hill,doctor,emergency,1


## 7. Clean `services_weekly.csv`

In [8]:
services_weekly = services_weekly.drop_duplicates()
services_weekly = services_weekly.drop_duplicates(subset=['week', 'service'], keep='first')

numeric_cols = ['available_beds', 'patients_request', 'patients_admitted',
                 'patients_refused', 'patient_satisfaction', 'staff_morale']
for col in numeric_cols:
    services_weekly[col] = pd.to_numeric(services_weekly[col], errors='coerce')
    services_weekly[col] = services_weekly[col].fillna(services_weekly[col].median())

# Missing event -> 'none' (no special event that week)
services_weekly['event'] = services_weekly['event'].fillna('none').astype(str).str.strip().str.lower()

services_weekly = services_weekly.dropna(subset=['week', 'service'])

print("Cleaned services_weekly shape:", services_weekly.shape)
services_weekly.head()

Cleaned services_weekly shape: (208, 10)


,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event
0,1,1,emergency,32,76,32,44,67,70,none
1,1,1,surgery,45,130,45,85,83,78,flu
2,1,1,general_medicine,37,201,37,164,97,43,flu
3,1,1,icu,22,31,22,9,84,91,flu
4,2,1,emergency,28,169,28,141,75,64,none


## 8. Normalize healthcare indicators
Derive a few KPI-ready fields so Tableau doesn't need calculated fields for the basics.

In [9]:
# Occupancy proxy at weekly-service level: admitted / available_beds
services_weekly['occupancy_rate'] = (
    services_weekly['patients_admitted'] / services_weekly['available_beds']
).replace([np.inf, -np.inf], np.nan).fillna(0).round(4)

# Refusal rate: refused / requested
services_weekly['refusal_rate'] = (
    services_weekly['patients_refused'] / services_weekly['patients_request']
).replace([np.inf, -np.inf], np.nan).fillna(0).round(4)

services_weekly[['week','service','occupancy_rate','refusal_rate']].head()

,week,service,occupancy_rate,refusal_rate
0,1,emergency,1.0,0.5789
1,1,surgery,1.0,0.6538
2,1,general_medicine,1.0,0.8159
3,1,icu,1.0,0.2903
4,2,emergency,1.0,0.8343


## 9. Final completeness check
Target from project evaluation criteria: **<2% missing values**, **>95% dataset completeness**.

In [10]:
for df, name in [(patients, "patients"), (staff, "staff"),
                  (staff_schedule, "staff_schedule"), (services_weekly, "services_weekly")]:
    missing_pct = df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100
    print(f"{name}: {missing_pct:.2f}% missing, {df.duplicated().sum()} duplicate rows remaining")

patients: 0.00% missing, 0 duplicate rows remaining
staff: 0.00% missing, 0 duplicate rows remaining
staff_schedule: 0.00% missing, 0 duplicate rows remaining
services_weekly: 0.00% missing, 0 duplicate rows remaining


## 10. Export Tableau-ready cleaned datasets

In [11]:
patients.to_csv(f"{PROCESSED_DIR}/patients_cleaned.csv", index=False)
staff.to_csv(f"{PROCESSED_DIR}/staff_cleaned.csv", index=False)
staff_schedule.to_csv(f"{PROCESSED_DIR}/staff_schedule_cleaned.csv", index=False)
services_weekly.to_csv(f"{PROCESSED_DIR}/services_weekly_cleaned.csv", index=False)

print("All 4 cleaned datasets exported to", PROCESSED_DIR)

All 4 cleaned datasets exported to ../data/processed


## Notes / Known limitations
- **Readmission Rate**: not directly available (no repeat-visit flag). If needed, derive by checking repeated `patient_id` — but this dataset generates a new `patient_id` per row, so treat as **out of scope** and document accordingly.
- **Multi-hospital comparison**: dataset represents a single hospital (no `hospital_id`/`hospital_name` column). If the Hospital Overview dashboard needs multi-hospital view, either drop that visual or add a constant `hospital_name` column.
- Linking key across all 4 cleaned tables: **`service`**. `staff_id` links `staff_cleaned` and `staff_schedule_cleaned`. `week` links `staff_schedule_cleaned` and `services_weekly_cleaned`.